# Mini-FORESIGHT — Step 9: Inventory Recommendations

In the previous step we identified **inventory risk** for each SKU.

Now we turn that analysis into **practical reorder recommendations**.

This notebook combines:
- `forecast_results.csv` — expected near-term demand
- `inventory_risk.csv` — current stock, lead-time demand, risk level
- `sku_master_clean.csv` — product details and lead times

For each SKU we will determine:
- Current stock
- Forecast demand
- Lead-time demand
- Safety stock
- Reorder point
- Recommended reorder quantity
- Urgency
- Recommendation / reason

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully")

In [ ]:
forecast = pd.read_csv(
    "../data/processed/forecast_results.csv"
)

risk = pd.read_csv(
    "../data/processed/inventory_risk.csv"
)

sku_master = pd.read_csv(
    "../data/processed/sku_master_clean.csv"
)

print("=== Dataset Shapes ===")
print("Forecast:", forecast.shape)
print("Inventory risk:", risk.shape)
print("SKU master:", sku_master.shape)

## Understanding the Inputs

**Forecast results** (`forecast_results.csv`)
- Contains the 3-day demand forecast per SKU.
- We aggregate it to get total forecast demand over the next 3 days.

**Inventory risk** (`inventory_risk.csv`)
- Contains current stock, average daily demand, lead-time demand, and risk level per SKU.

**SKU master** (`sku_master_clean.csv`)
- Contains product names, categories, prices, and supplier lead times.

We merge these to build the recommendation table.

In [ ]:
# Aggregate forecast to 3-day total per SKU
forecast_summary = (
    forecast.groupby("sku_id")["forecast_units"]
    .sum()
    .reset_index()
    .rename(columns={"forecast_units": "forecast_3_day_demand"})
)

print("=== 3-Day Forecast Demand ===")
print(forecast_summary.to_string(index=False))

## Build the Recommendation Base Table

We merge the inventory risk table with the forecast summary.

The inventory risk table already contains `product_name`, `category`, and `lead_time_days`, so we use it directly as the base.

In [ ]:
# inventory_risk.csv already contains forecast_3_day_demand, so drop it
# before merging to avoid duplicate columns
risk = risk.drop(columns=["forecast_3_day_demand"])

rec = risk.merge(
    forecast_summary,
    on="sku_id",
    how="left"
)

print("=== Recommendation Base Table ===")
base_display = rec[[
    "sku_id", "product_name", "current_stock",
    "forecast_3_day_demand", "lead_time_demand", "risk_level"
]].copy()
base_display["forecast_3_day_demand"] = base_display["forecast_3_day_demand"].round(2)
base_display["lead_time_demand"] = base_display["lead_time_demand"].round(2)
print(base_display.to_string(index=False))

## Safety Stock

**Safety stock** is a buffer of extra inventory to protect against unexpected demand or supply delays.

For this project we use a simple rule:

```
safety_stock = 20% of lead_time_demand
```

This is a simple, explainable buffer for a beginner project — not a universal standard.

In [ ]:
rec["safety_stock"] = rec["lead_time_demand"] * 0.2

print("=== Safety Stock ===")
safety_display = rec[[
    "sku_id", "lead_time_demand", "safety_stock"
]].copy()
safety_display["lead_time_demand"] = safety_display["lead_time_demand"].round(2)
safety_display["safety_stock"] = safety_display["safety_stock"].round(2)
print(safety_display.to_string(index=False))

## Reorder Point

The **reorder point** is the stock level at which a new order should be placed.

It is the amount of stock needed to cover demand during the supplier lead time, plus a safety buffer:

```
reorder_point = lead_time_demand + safety_stock
```

If current stock falls below the reorder point, it is time to reorder.

In [ ]:
rec["reorder_point"] = rec["lead_time_demand"] + rec["safety_stock"]

print("=== Reorder Point ===")
reorder_display = rec[[
    "sku_id", "lead_time_demand", "safety_stock", "reorder_point"
]].copy()
reorder_display["lead_time_demand"] = reorder_display["lead_time_demand"].round(2)
reorder_display["safety_stock"] = reorder_display["safety_stock"].round(2)
reorder_display["reorder_point"] = reorder_display["reorder_point"].round(2)
print(reorder_display.to_string(index=False))

## Recommended Reorder Quantity

The **recommended reorder quantity** is how much we should order to bring stock back up to the reorder point.

```
recommended_reorder_qty = reorder_point - current_stock
```

We clip the value at 0 so we never recommend a negative order.

This is a simple, practical rule for this project.

In [ ]:
rec["recommended_reorder_qty"] = (
    rec["reorder_point"] - rec["current_stock"]
).clip(lower=0)

print("=== Recommended Reorder Quantity ===")
qty_display = rec[[
    "sku_id", "current_stock", "reorder_point", "recommended_reorder_qty"
]].copy()
qty_display["reorder_point"] = qty_display["reorder_point"].round(2)
qty_display["recommended_reorder_qty"] = qty_display["recommended_reorder_qty"].round(2)
print(qty_display.to_string(index=False))

## Urgency and Recommendation

We classify **urgency** using a simple, explainable rule:

```
HIGH:   current_stock < lead_time_demand
MEDIUM: lead_time_demand <= current_stock < reorder_point
LOW:    current_stock >= reorder_point
```

We also generate a plain-language **recommendation** for each SKU based on the same logic.

In [ ]:
def classify_urgency(row):
    if row["current_stock"] < row["lead_time_demand"]:
        return "High"
    elif row["current_stock"] < row["reorder_point"]:
        return "Medium"
    else:
        return "Low"

rec["urgency"] = rec.apply(classify_urgency, axis=1)

def build_recommendation(row):
    if row["current_stock"] < row["lead_time_demand"]:
        return "Stock below lead-time demand - reorder immediately"
    elif row["current_stock"] < row["reorder_point"]:
        return "Stock below reorder point - plan a reorder"
    else:
        return "Stock above reorder point - no reorder needed"

rec["recommendation"] = rec.apply(build_recommendation, axis=1)

print("=== Urgency and Recommendation ===")
urgency_display = rec[[
    "sku_id", "current_stock", "lead_time_demand",
    "reorder_point", "urgency", "recommendation"
]].copy()
urgency_display["lead_time_demand"] = urgency_display["lead_time_demand"].round(2)
urgency_display["reorder_point"] = urgency_display["reorder_point"].round(2)
print(urgency_display.to_string(index=False))

## Final Recommendation Table

We assemble the complete recommendation table with all the key columns, sorted by urgency (High → Medium → Low).

In [ ]:
rec_final = rec[[
    "sku_id",
    "product_name",
    "category",
    "current_stock",
    "forecast_3_day_demand",
    "lead_time_demand",
    "safety_stock",
    "reorder_point",
    "recommended_reorder_qty",
    "urgency",
    "recommendation"
]].copy()

urgency_order = {"High": 0, "Medium": 1, "Low": 2}
rec_final["_order"] = rec_final["urgency"].map(urgency_order)
rec_final = rec_final.sort_values("_order").drop(columns="_order").reset_index(drop=True)

print("=== FINAL REORDER RECOMMENDATIONS ===")
rec_final_display = rec_final.copy()
for col in ["forecast_3_day_demand", "lead_time_demand", "safety_stock",
            "reorder_point", "recommended_reorder_qty"]:
    rec_final_display[col] = rec_final_display[col].round(2)
print(rec_final_display.to_string(index=False))

## Visualizing the Recommendations

A bar chart shows the recommended reorder quantity for each SKU, making it easy to see which products need the most stock.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(rec_final["sku_id"], rec_final["recommended_reorder_qty"])
plt.title("Recommended Reorder Quantity by SKU")
plt.xlabel("SKU")
plt.ylabel("Recommended Reorder Quantity")
plt.tight_layout()
plt.show()

## Save the Recommendations

We save the final recommendation table so it can be used by the Streamlit dashboard later.

In [ ]:
output_path = Path(
    "../data/processed/recommendations.csv"
)

rec_final.to_csv(
    output_path,
    index=False
)

print("Recommendations saved successfully:")
print(output_path)

In [ ]:
verification = pd.read_csv(
    "../data/processed/recommendations.csv"
)

print("=== Saved Recommendations Verification ===")
print("Shape:", verification.shape)
print()
print("Columns:", verification.columns.tolist())
print()
print("Missing values:")
print(verification.isna().sum())
print()
print("Data:")
print(verification.to_string(index=False))
print()
print("Expected 3 SKU rows:", len(verification) == 3)
print("Zero missing values:", verification.isna().sum().sum() == 0)
print("Urgency populated for every SKU:", verification["urgency"].notna().all())

# Recommendations Summary

The recommendations stage combines:
- Forecast demand
- Current inventory
- Lead-time demand
- Safety stock

to produce practical **reorder recommendations** for each SKU.

For each SKU we calculated:
- Safety stock (20% of lead-time demand)
- Reorder point (lead-time demand + safety stock)
- Recommended reorder quantity (reorder point − current stock, floored at 0)
- Urgency (High / Medium / Low)
- A plain-language recommendation

The output file is:

```
data/processed/recommendations.csv
```

The next pipeline step is:

---

## Step 10 — Streamlit Dashboard

That step will build an interactive dashboard to visualize the forecasts, WAPE evaluation, inventory risk, and recommendations.